In [ ]:
# Install pip packages
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade --no-cache-dir "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn "torch==2.6" torchvision boto3 pandas

In [ ]:
# Prepare dataset
!rm -rf dataset/ dataset.zip
!wget -nc -q -O "./dataset.zip" 'https://www.kaggle.com/api/v1/datasets/download/uciml/iris'
!unzip -q dataset.zip

In [ ]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [ ]:
# Upload dataset zip file to S3
from sagemaker.s3 import S3Uploader

inputs = S3Uploader.upload("dataset.zip", "s3://{}/{}".format(bucket, prefix))

In [ ]:
import pandas as pd

df = pd.read_csv('./Iris.csv')
print(df.shape)
print(df.isnull().sum())
df.head()

In [ ]:
from sagemaker.xgboost import XGBoost

estimator = XGBoost(
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="3.0-5",
    hyperparameters={
      "n_estimators": 100,
      "max_depth": 5,
      "learning_rate": 0.05
    },
)

estimator.fit(inputs={"train": f's3://{bucket}/{prefix}/dataset.zip'}, wait=True)

In [ ]:
# Create Endpoint
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
)

In [ ]:
# Create new model + Update Endpoint
import boto3
import time

sm = boto3.client("sagemaker")

model = estimator.create_model()
model_name = model.name
model._create_sagemaker_model(instance_type="ml.m5.large")

new_config = f'endpoint-configuration-{int(time.time())}'
sm.create_endpoint_config(
    EndpointConfigName=new_config,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.m5.large"
    }]
)

sm.update_endpoint(
    EndpointName = "<ENDPOINT_NAME>",
    EndpointConfigName = new_config
)

In [ ]:
# Inference test — 1D (단일 샘플)
# input_fn 안에서 자동으로 (1, n)으로 reshape 됨
import json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = '<ENDPOINT_NAME>'

body = {"instance": [5.1, 3.5, 1.4, 0.2]}
# body = {"instance": [
#     [5.1, 3.5, 1.4, 0.2],
#     [6.7, 3.0, 5.2, 2.3],
# ]}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(body),
)
result = json.load(response["Body"])
print(result)                   # {'class_name': ['Iris-setosa']}
print(result['class_name'])     # ['Iris-setosa']
